In [1]:
# The following code will only execute
# successfully when compression is complete

import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohabenloughmari/miccai-task2-full")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'miccai-task2-full' dataset.
Path to dataset files: /kaggle/input/miccai-task2-full


In [2]:
!git clone https://github.com/j-morano/MIRAGE.git
import sys
sys.path.append('/content/MIRAGE')  # Adjust path if not using Colab

fatal: destination path 'MIRAGE' already exists and is not an empty directory.


In [3]:
!ls /content/MIRAGE/hf/

helper_hf.py  mirage_hf.py  README.md


In [4]:
import pandas as pd
import torch
from tqdm import tqdm
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import sys
import numpy as np
from functools import partial
from collections import OrderedDict
from sklearn.metrics import (
    f1_score, cohen_kappa_score, recall_score, accuracy_score,
    classification_report, precision_score, roc_auc_score,
    confusion_matrix, matthews_corrcoef
)
from sklearn.preprocessing import label_binarize
from torch.utils.data import WeightedRandomSampler
import torch.optim as optim
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download

sys.path.append('/content/MIRAGE')
from mirage.input_adapters import PatchedInputAdapter
from mirage.model import MIRAGEModel
from mirage.utils import pair

import warnings
warnings.filterwarnings("ignore")

# ============================================================
# 1. Build MIRAGE from scratch + load safetensors weights
# ============================================================

def build_mirage_base(input_size=512, patch_size=32, modalities='bscan-slo', device='cuda'):
    """Build MIRAGE-Base model manually and load HF weights."""

    in_domains = modalities.split('-')
    input_size = pair(input_size)
    patch_size_pair = pair(patch_size)

    # Build a minimal args namespace (mimicking what the .pth would contain)
    class Args:
        pass
    args = Args()
    args.in_domains = in_domains
    args.out_domains = in_domains  # not used since we disable output adapters
    args.model = 'miragepre_base'
    args.num_global_tokens = 1
    args.drop_path = 0.0
    args.decoder_dim = 768
    args.decoder_depth = 2
    args.decoder_num_heads = 12
    args.decoder_use_task_queries = True
    args.decoder_use_xattn = True
    args.patch_size = {}
    args.input_size = {}
    args.grid_size = {}
    args.grid_sizes = {}

    for domain in in_domains:
        args.patch_size[domain] = patch_size_pair
        args.input_size[domain] = input_size
        gs = (input_size[0] // patch_size_pair[0], input_size[1] // patch_size_pair[1])
        args.grid_size[domain] = list(gs)
        args.grid_sizes[domain] = list(gs)

    # Build input adapters
    input_adapters = {
        domain: PatchedInputAdapter(
            num_channels=1,
            stride_level=1,
            patch_size_full=patch_size_pair,
            image_size=input_size,
        )
        for domain in in_domains
    }

    # Create model (ViT-Base: dim=768, depth=12, heads=12)
    model = MIRAGEModel(
        args=args,
        input_adapters=input_adapters,
        output_adapters=None,  # We only need encoder features
        num_global_tokens=1,
        dim_tokens=768,
        depth=12,
        num_heads=12,
        mlp_ratio=4.0,
        qkv_bias=True,
        drop_path_rate=0.0,
    )

    # Download and load weights
    sf_path = hf_hub_download(repo_id="j-morano/MIRAGE-Base", filename="model.safetensors")
    state_dict = load_file(sf_path)

    # Remove "model." prefix since we're loading into MIRAGEModel directly
    cleaned = {}
    for k, v in state_dict.items():
        if k.startswith("model."):
            cleaned[k[len("model."):]] = v
        else:
            cleaned[k] = v

    # Filter out output adapter keys and any keys not in our model
    model_keys = set(model.state_dict().keys())
    filtered = {k: v for k, v in cleaned.items() if k in model_keys}

    msg = model.load_state_dict(filtered, strict=False)
    print(f"  Loaded {len(filtered)} keys")
    print(f"  Missing keys: {len(msg.missing_keys)}")
    print(f"  Unexpected keys: {len(msg.unexpected_keys)}")
    if msg.missing_keys:
        print(f"  Missing: {msg.missing_keys[:5]}...")

    return model, args


class MIRAGEClassifier(nn.Module):
    def __init__(self, num_classes, input_size=512, patch_size=32,
                 modalities='bscan-slo', device='cuda', freeze_backbone=True):
        super().__init__()

        self.mirage_model, self.args = build_mirage_base(
            input_size=input_size,
            patch_size=patch_size,
            modalities=modalities,
            device=device,
        )
        self.modalities = modalities.split('-')
        self.num_global_tokens = self.args.num_global_tokens

        # Get embed dim
        self.embed_dim = self.mirage_model.encoder[0].norm1.normalized_shape[0]
        print(f"Embedding dimension: {self.embed_dim}")

        if freeze_backbone:
            for param in self.mirage_model.parameters():
                param.requires_grad = False

        self.norm = nn.LayerNorm(self.embed_dim)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    def forward(self, bscan, slo):
        """
        Args:
            bscan: [B, 1, 512, 512] grayscale tensor in [0, 1]
            slo:   [B, 1, 512, 512] grayscale tensor in [0, 1]
        """
        x_dict = {}
        if 'bscan' in self.modalities:
            x_dict['bscan'] = bscan
        if 'slo' in self.modalities:
            x_dict['slo'] = slo

        # Forward through encoder (output_adapters=None returns encoder tokens)
        encoder_tokens, _masks = self.mirage_model(x_dict, mask_inputs=False)
        # encoder_tokens: [B, num_patch_tokens + num_global_tokens, embed_dim]

        # Pool patch tokens (exclude global tokens at the end)
        patch_features = encoder_tokens[:, :-self.num_global_tokens, :].mean(dim=1)
        patch_features = self.norm(patch_features)
        logits = self.classifier(patch_features)
        return logits


# ============================================================
# 2. Dataset (grayscale, [0,1], 1 channel for MIRAGE)
# ============================================================

base_path = path  # Set your base path
task_path = os.path.join(base_path, 'Task_2')

df_train = pd.read_csv(os.path.join(task_path, "df_task2_train.csv"))
df_val = pd.read_csv(os.path.join(task_path, "df_task2_val.csv"))
df_test = pd.read_csv(os.path.join(task_path, "df_task2_test.csv"))

device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')


class CustomImageDataset(Dataset):
    def __init__(self, dataframe, root_dir, input_size=512, augment=False):
        self.dataframe = dataframe
        self.root_dir = root_dir

        # MIRAGE expects: grayscale, 1 channel, [0, 1] range
        if augment:
            self.transform = transforms.Compose([
                transforms.Resize((input_size, input_size)),
                transforms.Grayscale(num_output_channels=1),
                transforms.RandomHorizontalFlip(p=0.4),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                transforms.ToTensor(),  # -> [0, 1]
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((input_size, input_size)),
                transforms.Grayscale(num_output_channels=1),
                transforms.ToTensor(),
            ])

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['image'])
        localiser_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['LOCALIZER'])

        image = Image.open(img_name).convert('RGB')
        localiser = Image.open(localiser_name).convert('RGB')

        image = self.transform(image)       # [1, 512, 512]
        localiser = self.transform(localiser)  # [1, 512, 512]

        label = self.dataframe.iloc[idx]['label']
        return image, localiser, label


batch_size = 4  # 512x512 uses more VRAM

train_ds = CustomImageDataset(df_train, os.path.join(task_path, 'train'), augment=True)
val_ds = CustomImageDataset(df_val, os.path.join(task_path, 'val'), augment=False)
test_ds = CustomImageDataset(df_test, os.path.join(task_path, 'test'), augment=False)

labels = df_train['label'].values.astype(int)
class_counts = np.bincount(labels)
sample_weights = 1.0 / class_counts[labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

nb_classes = df_train['label'].nunique()


# ============================================================
# 3. Helpers
# ============================================================

def specificity_per_class(conf_matrix):
    specificities = []
    for i in range(len(conf_matrix)):
        tn = conf_matrix.sum() - (conf_matrix[i, :].sum() + conf_matrix[:, i].sum() - conf_matrix[i, i])
        fp = conf_matrix[:, i].sum() - conf_matrix[i, i]
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specificities.append(spec)
    return specificities

def overall_accuracy(conf_matrix):
    return np.trace(conf_matrix) / conf_matrix.sum()


# ============================================================
# 4. Training Function
# ============================================================

def train(epochs, net, train_loader, test_loader, optimizer, scheduler, loss_function, device):
    best_acc = 0.0
    for epoch in range(epochs):
        net.train()
        running_loss = 0.0
        train_bar = tqdm(train_loader, file=sys.stdout)

        for step, datax in enumerate(train_bar):
            images, localizer, labels = datax
            images = images.to(device).float()
            localizer = localizer.to(device).float()
            labels = labels.to(device).long()

            optimizer.zero_grad()
            outputs = net(images, localizer)
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()
            train_bar.desc = f"train epoch[{epoch + 1}/{epochs}] loss:{loss:.3f}"

        net.eval()
        all_preds, all_labels, all_probs = [], [], []
        acc = 0.0

        with torch.no_grad():
            val_bar = tqdm(test_loader, file=sys.stdout)
            for val_data in val_bar:
                val_images, val_localizer, val_labels = val_data
                val_images = val_images.to(device).float()
                val_localizer = val_localizer.to(device).float()

                outputs = net(val_images, val_localizer)
                probs = torch.softmax(outputs, dim=1)
                predict_y = torch.max(probs, dim=1)[1]

                all_preds.extend(predict_y.cpu().numpy())
                all_labels.extend(val_labels.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())
                acc += torch.eq(predict_y, val_labels.to(device)).sum().item()

        val_accurate = acc / len(test_loader.dataset)
        precision = precision_score(all_labels, all_preds, average='weighted')
        recall = recall_score(all_labels, all_preds, average='weighted')
        f1 = f1_score(all_labels, all_preds, average='weighted')
        rk_corr = matthews_corrcoef(all_labels, all_preds)
        qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')

        conf_matrix = confusion_matrix(all_labels, all_preds)
        specificity = specificity_per_class(conf_matrix)
        avg_specificity = sum(specificity) / len(specificity)
        overall_acc = overall_accuracy(conf_matrix)

        n_classes = len(conf_matrix)
        all_labels_one_hot = label_binarize(all_labels, classes=list(range(n_classes)))
        try:
            auc = roc_auc_score(all_labels_one_hot, all_probs, multi_class='ovr')
        except ValueError:
            auc = float('nan')

        print(f'[epoch {epoch + 1}] train_loss: {running_loss / len(train_loader):.3f} '
              f'val_accuracy: {val_accurate:.4f} precision: {precision:.4f} '
              f'recall: {recall:.4f} specificity: {avg_specificity:.4f} '
              f'f1_score: {f1:.4f} auc: {auc:.4f} overall_accuracy: {overall_acc:.4f} '
              f'mcc: {rk_corr:.4f} qwk: {qwk:.4f}')
        print(f'lr: {scheduler.get_last_lr()[-1]:.8f}')
        print('\nClassification Report:')
        print(classification_report(all_labels, all_preds))

        if val_accurate > best_acc:
            print('\nSaving checkpoint...')
            best_acc = val_accurate
            state = {
                'model': net.state_dict(),
                'optimizer': optimizer.state_dict(),
                'lr_scheduler': scheduler.state_dict(),
                'acc': best_acc,
                'epoch': epoch,
            }
            torch.save(state, 'state.pt')

    print('Finished Training')


# ============================================================
# 5. Run
# ============================================================

model = MIRAGEClassifier(
    num_classes=nb_classes,
    input_size=512,
    patch_size=32,
    modalities='bscan-slo',
    device=str(device),
    freeze_backbone=True,
).to(device)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=1e-2
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=len(train_loader) * 30)

class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32)
class_weights = class_weights / class_weights.sum()
loss_function = nn.CrossEntropyLoss(weight=class_weights.to(device))

train(
    epochs=30,
    net=model,
    train_loader=train_loader,
    test_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    loss_function=loss_function,
    device=device,
)

Image size: (512, 512), Patch size: (32, 32)
Image size: (512, 512), Patch size: (32, 32)
  Loaded 151 keys
  Missing keys: 0
  Unexpected keys: 0
Embedding dimension: 768
  0%|          | 0/2021 [00:00<?, ?it/s]bscan 256
slo 256
> Token distribution: {'bscan': 0.5, 'slo': 0.5}
100%|██████████| 956/956 [05:38<00:00,  2.83it/s]
[epoch 1] train_loss: 0.693 val_accuracy: 0.1342 precision: 0.0323 recall: 0.1342 specificity: 0.6606 f1_score: 0.0519 auc: 0.4689 overall_accuracy: 0.1342 mcc: -0.0401 qwk: -0.0539
lr: 0.00009973

Classification Report:
              precision    recall  f1-score   support

         0.0       0.05      0.14      0.07       351
         1.0       0.00      0.00      0.00      2821
         2.0       0.16      0.71      0.27       650

    accuracy                           0.13      3822
   macro avg       0.07      0.28      0.11      3822
weighted avg       0.03      0.13      0.05      3822


Saving checkpoint...
train epoch[2/30] loss:0.358:   9%|▉         | 

KeyboardInterrupt: 

In [ ]:
from huggingface_hub import list_repo_files
files = list_repo_files("j-morano/MIRAGE-Base")
print(files)


In [ ]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import torch

# Download the safetensors file
sf_path = hf_hub_download(repo_id="j-morano/MIRAGE-Base", filename="model.safetensors")
print(f"Downloaded to: {sf_path}")

# Load and inspect what's inside
state = load_file(sf_path)
print(f"Number of keys: {len(state)}")
print("First 10 keys:", list(state.keys())[:10])

In [ ]:
import json
config_path = hf_hub_download(repo_id="j-morano/MIRAGE-Base", filename="config.json")
with open(config_path) as f:
    config = json.load(f)
print(json.dumps(config, indent=2))

In [ ]:
!cat /content/MIRAGE/mirage/model.py

In [ ]:
print("All keys:")
for k in sorted(state.keys()):
    print(f"  {k}: {state[k].shape}")

In [ ]:
import pandas as pd
import torch
from tqdm import tqdm
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import sys
import numpy as np
from sklearn.metrics import (
    f1_score, cohen_kappa_score, recall_score, accuracy_score,
    classification_report, precision_score, roc_auc_score,
    confusion_matrix, matthews_corrcoef
)
from sklearn.preprocessing import label_binarize
from torch.utils.data import WeightedRandomSampler
import torch.optim as optim

sys.path.append('/content/MIRAGE')
from mirage_wrapper import MIRAGEWrapper, MIRAGEClsGlobal, MIRAGEClsCLS, MIRAGEClsTokenMix

import warnings
warnings.filterwarnings("ignore")

# ============================================================
# 1. Download MIRAGE weights
# ============================================================
# Download weights from HuggingFace manually
!mkdir -p /content/MIRAGE/__weights
!wget -q -O /content/MIRAGE/__weights/MIRAGE-Base.pth "https://huggingface.co/j-morano/MIRAGE-Base/resolve/main/model.safetensors" || true

# If the above doesn't work, try downloading the .pth file directly:
# Check what files are on the HF repo
from huggingface_hub import hf_hub_download
weights_path = hf_hub_download(repo_id="j-morano/MIRAGE-Base", filename="MIRAGE-Base.pth")
print(f"Weights downloaded to: {weights_path}")

In [ ]:
import pandas as pd
import torch
from tqdm import tqdm
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
from sklearn.metrics import (
    f1_score, cohen_kappa_score, recall_score, accuracy_score,
    classification_report, precision_score, roc_auc_score,
    confusion_matrix, matthews_corrcoef
)
from sklearn.preprocessing import label_binarize
from scipy.stats import spearmanr
from torch.utils.data import WeightedRandomSampler
from huggingface_hub import PyTorchModelHubMixin
import sys
import torch.optim as optim

import warnings
warnings.filterwarnings("ignore")

# ============================================================
# 1. MIRAGE Model with Classification Head (dual-input: B-scan + SLO)
# ============================================================

class MIRAGEhf(MIRAGEWrapper, PyTorchModelHubMixin):
    def __init__(
        self,
        input_size=512,
        patch_size=32,
        modalities='bscan-slo',
        size='base',
    ):
        super().__init__(
            input_size=input_size,
            patch_size=patch_size,
            modalities=modalities,
            size=size,
        )


class MIRAGEClassifier(nn.Module):
    """
    Wraps the pretrained MIRAGE encoder and adds a classification head.
    MIRAGE expects two inputs: B-scan (OCT image) and SLO (localizer/fundus).
    """
    def __init__(self, num_classes, mirage_size='base', freeze_backbone=True):
        super().__init__()

        # Load pretrained MIRAGE
        if mirage_size == 'base':
            self.backbone = MIRAGEhf.from_pretrained("j-morano/MIRAGE-Base")
            embed_dim = 768   # ViT-Base hidden size
        elif mirage_size == 'large':
            self.backbone = MIRAGEhf.from_pretrained("j-morano/MIRAGE-Large")
            embed_dim = 1024  # ViT-Large hidden size
        else:
            raise ValueError(f"Unknown size: {mirage_size}")

        # Optionally freeze backbone
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        # Classification head on top of the [CLS] token embedding
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.3),
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    def forward(self, bscan, slo):
        """
        Args:
            bscan: OCT B-scan image tensor [B, C, H, W]
            slo:   SLO/localizer image tensor [B, C, H, W]
        Returns:
            logits: [B, num_classes]
        """
        # MIRAGE encodes both modalities; returns embeddings
        # Adjust this call based on actual MIRAGEWrapper.forward() signature
        features = self.backbone(bscan, slo)  # Expected shape: [B, embed_dim]

        # If features is a tuple/dict, extract the pooled representation
        if isinstance(features, dict):
            features = features.get('pooled', features.get('cls', list(features.values())[0]))
        elif isinstance(features, (tuple, list)):
            features = features[0]

        # If we get sequence output [B, seq_len, dim], take the CLS token
        if features.dim() == 3:
            features = features[:, 0, :]

        logits = self.classifier(features)
        return logits


# ============================================================
# 2. Dataset & DataLoaders
# ============================================================

base_path = path  # Set your base path
path = os.path.join(base_path, 'Task_2')

df_train = pd.read_csv(os.path.join(path, "df_task2_train.csv"))
df_val = pd.read_csv(os.path.join(path, "df_task2_val.csv"))
df_test = pd.read_csv(os.path.join(path, "df_task2_test.csv"))

device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')


class CustomImageDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['image'])
        localiser_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['LOCALIZER'])

        image = Image.open(img_name).convert('RGB')
        localiser = Image.open(localiser_name).convert('RGB')

        if self.transform:
            image = self.transform(image)
            localiser = self.transform(localiser)

        label = self.dataframe.iloc[idx]['label']
        return image, localiser, label

# MIRAGE expects 512x512 input by default
train_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.AugMix(alpha=0.4),
    transforms.RandomHorizontalFlip(p=0.4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

batch_size = 8  # Adjust based on GPU memory

train_ds = CustomImageDataset(df_train, os.path.join(path, 'train'), transform=train_transform)
val_ds = CustomImageDataset(df_val, os.path.join(path, 'val'), transform=test_transform)
test_ds = CustomImageDataset(df_test, os.path.join(path, 'test'), transform=test_transform)

labels = df_train['label'].values.astype(int)
class_counts = np.bincount(labels)
sample_weights = 1.0 / class_counts[labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

nb_classes = df_train['label'].nunique()


# ============================================================
# 3. Helper Functions
# ============================================================

def specificity_per_class(conf_matrix):
    specificities = []
    for i in range(len(conf_matrix)):
        tn = conf_matrix.sum() - (conf_matrix[i, :].sum() + conf_matrix[:, i].sum() - conf_matrix[i, i])
        fp = conf_matrix[:, i].sum() - conf_matrix[i, i]
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specificities.append(spec)
    return specificities


def overall_accuracy(conf_matrix):
    return np.trace(conf_matrix) / conf_matrix.sum()


# ============================================================
# 4. Training Function (adapted for dual-input MIRAGE)
# ============================================================

state = {}

def train(epochs, net, train_loader, test_loader, optimizer, scheduler, loss_function, device):
    best_acc = 0.0
    for epoch in range(epochs):
        net.train()
        running_loss = 0.0
        train_bar = tqdm(train_loader, file=sys.stdout)

        # --- Training Loop ---
        for step, datax in enumerate(train_bar):
            images, localizer, labels = datax
            images = images.to(device).float()
            localizer = localizer.to(device).float()  # SLO/localizer as second input
            labels = labels.to(device).long()

            optimizer.zero_grad()
            outputs = net(images, localizer)  # Pass both modalities
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()
            train_bar.desc = f"train epoch[{epoch + 1}/{epochs}] loss:{loss:.3f}"

        # --- Validation Loop ---
        net.eval()
        all_preds = []
        all_labels = []
        all_probs = []
        acc = 0.0

        with torch.no_grad():
            val_bar = tqdm(test_loader, file=sys.stdout)
            for val_data in val_bar:
                val_images, val_localizer, val_labels = val_data  # Unpack 3 values
                val_images = val_images.to(device).float()
                val_localizer = val_localizer.to(device).float()

                outputs = net(val_images, val_localizer)  # Pass both modalities
                probs = torch.softmax(outputs, dim=1)
                predict_y = torch.max(probs, dim=1)[1]

                all_preds.extend(predict_y.cpu().numpy())
                all_labels.extend(val_labels.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())
                acc += torch.eq(predict_y, val_labels.to(device)).sum().item()

        # --- Metrics ---
        val_accurate = acc / len(test_loader.dataset)
        precision = precision_score(all_labels, all_preds, average='weighted')
        recall = recall_score(all_labels, all_preds, average='weighted')
        f1 = f1_score(all_labels, all_preds, average='weighted')
        rk_corr = matthews_corrcoef(all_labels, all_preds)
        qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')

        conf_matrix = confusion_matrix(all_labels, all_preds)
        specificity = specificity_per_class(conf_matrix)
        avg_specificity = sum(specificity) / len(specificity)
        overall_acc = overall_accuracy(conf_matrix)

        n_classes = len(conf_matrix)
        all_labels_one_hot = label_binarize(all_labels, classes=list(range(n_classes)))
        try:
            auc = roc_auc_score(all_labels_one_hot, all_probs, multi_class='ovr')
        except ValueError:
            auc = float('nan')

        print(f'[epoch {epoch + 1}] train_loss: {running_loss / len(train_loader):.3f} '
              f'val_accuracy: {val_accurate:.4f} precision: {precision:.4f} '
              f'recall: {recall:.4f} specificity: {avg_specificity:.4f} '
              f'f1_score: {f1:.4f} auc: {auc:.4f} overall_accuracy: {overall_acc:.4f} '
              f'mcc: {rk_corr:.4f} qwk: {qwk:.4f}')
        print(f'lr: {scheduler.get_last_lr()[-1]:.8f}')
        print('\nClassification Report:')
        print(classification_report(all_labels, all_preds))

        if val_accurate > best_acc:
            print('\nSaving checkpoint...')
            best_acc = val_accurate
            state = {
                'model': net.state_dict(),
                'optimizer': optimizer.state_dict(),
                'lr_scheduler': scheduler.state_dict(),
                'acc': best_acc,
                'epoch': epoch,
            }
            torch.save(state, 'state.pt')

    print('Finished Training')


# ============================================================
# 5. Run Training
# ============================================================

# Initialize model
model = MIRAGEClassifier(
    num_classes=nb_classes,
    mirage_size='base',       # or 'large'
    freeze_backbone=True       # Set False to fine-tune the full model
).to(device)

# Optimizer & Scheduler
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=len(train_loader) * 30)

# Loss with class weights for imbalanced data
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32)
class_weights = class_weights / class_weights.sum()
loss_function = nn.CrossEntropyLoss(weight=class_weights.to(device))

# Train
train(
    epochs=5,
    net=model,
    train_loader=train_loader,
    test_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    loss_function=loss_function,
    device=device,
)

In [ ]:
!cat /content/MIRAGE/mirage_wrapper.py

In [ ]:
# ============================================================
# 2. Custom Classifier wrapping MIRAGE
# ============================================================

class MIRAGEClassifier(nn.Module):
    """
    Uses MIRAGE as a dual-modality encoder (bscan + slo),
    extracts features, and adds a classification head.
    """
    def __init__(self, num_classes, weights_path, input_size=512, patch_size=32,
                 modalities='bscan-slo', device='cuda', freeze_backbone=True):
        super().__init__()

        self.backbone = MIRAGEWrapper(
            input_size=input_size,
            patch_size=patch_size,
            modalities=modalities,
            weights=weights_path,
            device=device,
        )
        # Remove output adapters to get raw encoder features
        self.backbone.model.output_adapters = None

        self.modalities = modalities.split('-')

        # Get embed dim from encoder
        self.embed_dim = self.backbone.model.encoder[0].norm1.normalized_shape[0]
        print(f"Embedding dimension: {self.embed_dim}")

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        self.norm = nn.LayerNorm(self.embed_dim)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    def forward(self, bscan, slo):
        """
        Args:
            bscan: [B, 1, H, W] grayscale tensor in [0, 1]
            slo:   [B, 1, H, W] grayscale tensor in [0, 1]
        Returns:
            logits: [B, num_classes]
        """
        x_dict = {}
        if 'bscan' in self.modalities:
            x_dict['bscan'] = bscan
        if 'slo' in self.modalities:
            x_dict['slo'] = slo

        # With output_adapters=None, model returns encoder features
        features, _masks = self.backbone.model(x_dict, mask_inputs=False)
        # features shape: [B, num_tokens, embed_dim]

        # Pool: mean of patch tokens (exclude global tokens)
        num_global = self.backbone.args.num_global_tokens
        patch_features = features[:, :-num_global, :].mean(dim=1)  # [B, embed_dim]

        patch_features = self.norm(patch_features)
        logits = self.classifier(patch_features)
        return logits


# ============================================================
# 3. Dataset (outputs grayscale [0,1] tensors for MIRAGE)
# ============================================================

base_path = path  # Set your base path
task_path = os.path.join(base_path, 'Task_2')

df_train = pd.read_csv(os.path.join(task_path, "df_task2_train.csv"))
df_val = pd.read_csv(os.path.join(task_path, "df_task2_val.csv"))
df_test = pd.read_csv(os.path.join(task_path, "df_task2_test.csv"))

device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')


class CustomImageDataset(Dataset):
    """
    MIRAGE expects grayscale images normalized to [0, 1],
    shape [B, 1, H, W]. NOT ImageNet normalization.
    """
    def __init__(self, dataframe, root_dir, input_size=512, augment=False):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.input_size = input_size
        self.augment = augment

        self.base_transform = transforms.Compose([
            transforms.Resize((input_size, input_size)),
            transforms.Grayscale(num_output_channels=1),  # MIRAGE expects 1 channel
            transforms.ToTensor(),  # Converts to [0, 1]
        ])

        self.aug_transform = transforms.Compose([
            transforms.Resize((input_size, input_size)),
            transforms.Grayscale(num_output_channels=1),
            transforms.RandomHorizontalFlip(p=0.4),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['image'])
        localiser_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['LOCALIZER'])

        image = Image.open(img_name).convert('RGB')
        localiser = Image.open(localiser_name).convert('RGB')

        if self.augment:
            image = self.aug_transform(image)
            localiser = self.aug_transform(localiser)
        else:
            image = self.base_transform(image)
            localiser = self.base_transform(localiser)

        label = self.dataframe.iloc[idx]['label']
        return image, localiser, label


batch_size = 4  # 512x512 images use more memory

train_ds = CustomImageDataset(df_train, os.path.join(task_path, 'train'), augment=True)
val_ds = CustomImageDataset(df_val, os.path.join(task_path, 'val'), augment=False)
test_ds = CustomImageDataset(df_test, os.path.join(task_path, 'test'), augment=False)

labels = df_train['label'].values.astype(int)
class_counts = np.bincount(labels)
sample_weights = 1.0 / class_counts[labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

nb_classes = df_train['label'].nunique()


# ============================================================
# 4. Helpers
# ============================================================

def specificity_per_class(conf_matrix):
    specificities = []
    for i in range(len(conf_matrix)):
        tn = conf_matrix.sum() - (conf_matrix[i, :].sum() + conf_matrix[:, i].sum() - conf_matrix[i, i])
        fp = conf_matrix[:, i].sum() - conf_matrix[i, i]
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specificities.append(spec)
    return specificities

def overall_accuracy(conf_matrix):
    return np.trace(conf_matrix) / conf_matrix.sum()
def train(epochs, net, train_loader, test_loader, optimizer, scheduler, loss_function, device):
    best_acc = 0.0
    for epoch in range(epochs):
        net.train()
        running_loss = 0.0
        train_bar = tqdm(train_loader, file=sys.stdout)

        for step, datax in enumerate(train_bar):
            images, localizer, labels = datax
            images = images.to(device).float()       
            localizer = localizer.to(device).float()  
            labels = labels.to(device).long()

            optimizer.zero_grad()
            outputs = net(images, localizer)  
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()
            train_bar.desc = f"train epoch[{epoch + 1}/{epochs}] loss:{loss:.3f}"
        net.eval()
        all_preds, all_labels, all_probs = [], [], []
        acc = 0.0
        with torch.no_grad():
            val_bar = tqdm(test_loader, file=sys.stdout)
            for val_data in val_bar:
                val_images, val_localizer, val_labels = val_data
                val_images = val_images.to(device).float()
                val_localizer = val_localizer.to(device).float()

                outputs = net(val_images, val_localizer)
                probs = torch.softmax(outputs, dim=1)
                predict_y = torch.max(probs, dim=1)[1]

                all_preds.extend(predict_y.cpu().numpy())
                all_labels.extend(val_labels.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())
                acc += torch.eq(predict_y, val_labels.to(device)).sum().item()

        val_accurate = acc / len(test_loader.dataset)
        precision = precision_score(all_labels, all_preds, average='weighted')
        recall = recall_score(all_labels, all_preds, average='weighted')
        f1 = f1_score(all_labels, all_preds, average='weighted')
        rk_corr = matthews_corrcoef(all_labels, all_preds)
        qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
        conf_matrix = confusion_matrix(all_labels, all_preds)
        specificity = specificity_per_class(conf_matrix)
        avg_specificity = sum(specificity) / len(specificity)
        overall_acc = overall_accuracy(conf_matrix)
        n_classes = len(conf_matrix)
        all_labels_one_hot = label_binarize(all_labels, classes=list(range(n_classes)))
        try:
            auc = roc_auc_score(all_labels_one_hot, all_probs, multi_class='ovr')
        except ValueError:
            auc = float('nan')

        print(f'[epoch {epoch + 1}] train_loss: {running_loss / len(train_loader):.3f} '
              f'val_accuracy: {val_accurate:.4f} precision: {precision:.4f} '
              f'recall: {recall:.4f} specificity: {avg_specificity:.4f} '
              f'f1_score: {f1:.4f} auc: {auc:.4f} overall_accuracy: {overall_acc:.4f} '
              f'mcc: {rk_corr:.4f} qwk: {qwk:.4f}')
        print(f'lr: {scheduler.get_last_lr()[-1]:.8f}')
        print('\nClassification Report:')
        print(classification_report(all_labels, all_preds))

        if val_accurate > best_acc:
            print('\nSaving checkpoint...')
            best_acc = val_accurate
            state = {
                'model': net.state_dict(),
                'optimizer': optimizer.state_dict(),
                'lr_scheduler': scheduler.state_dict(),
                'acc': best_acc,
                'epoch': epoch,
            }
            torch.save(state, 'state.pt')

    print('Finished Training')
model = MIRAGEClassifier(
    num_classes=nb_classes,
    weights_path=weights_path,  
    input_size=512,
    patch_size=32,
    modalities='bscan-slo',
    device=str(device),
    freeze_backbone=True,
).to(device)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=1e-2
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=len(train_loader) * 30)

class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32)
class_weights = class_weights / class_weights.sum()
loss_function = nn.CrossEntropyLoss(weight=class_weights.to(device))

train(
    epochs=30,
    net=model,
    train_loader=train_loader,
    test_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    loss_function=loss_function,
    device=device,
)

In [ ]:
import pandas as pd
import torch
from tqdm import tqdm
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
from sklearn.metrics import f1_score, cohen_kappa_score, recall_score, accuracy_score, classification_report
from scipy.stats import spearmanr
from torch.utils.data import WeightedRandomSampler

import warnings
warnings.filterwarnings("ignore")

base_path = path
path = os.path.join(base_path, 'Task_2')

df_train = pd.read_csv(os.path.join(path, "df_task2_train.csv"))
df_val = pd.read_csv(os.path.join(path, "df_task2_val.csv"))
df_test = pd.read_csv(os.path.join(path, "df_task2_test.csv"))

device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')


class CustomImageDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        #img_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['image'])
        localiser_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['LOCALIZER'])

        #image = Image.open(img_name).convert('RGB')
        localiser = Image.open(localiser_name).convert('RGB')

        if self.transform:
            #image = self.transform(image)
            localiser = self.transform(localiser)

        label = self.dataframe.iloc[idx]['label']
        return localiser, label


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.AugMix(alpha= 0.4),
    transforms.RandomHorizontalFlip(p=0.4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
train_ds = CustomImageDataset(df_train, os.path.join(path, 'train'), transform=train_transform)
val_ds = CustomImageDataset(df_val, os.path.join(path, 'val'), transform=test_transform)
test_ds = CustomImageDataset(df_test, os.path.join(path, 'test'), transform=test_transform)

labels = df_train['label'].values.astype(int)

class_counts = np.bincount(labels)
sample_weights = 1.0 / class_counts[labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

nb_classes = df_train['label'].nunique()



In [ ]:
%cd MedViTV2

In [ ]:
from MedViT import MedViT_tiny, MedViT_small, MedViT_base, MedViT_large
import requests
model_classes = {
    'MedViT_tiny': MedViT_tiny,
    'MedViT_small': MedViT_small,
    'MedViT_base': MedViT_base,
    'MedViT_large': MedViT_large
}

model_urls = {
    "MedViT_tiny": "https://dl.dropbox.com/scl/fi/496jbihqp360jacpji554/MedViT_tiny.pth?rlkey=6hb9froxugvtg8l639jmspxfv&st=p9ef06j8&dl=0",
    "MedViT_small": "https://dl.dropbox.com/scl/fi/6nnec8hxcn5da6vov7h2a/MedViT_small.pth?rlkey=yf5twra1cv6ep2oqr79tbzyg5&st=rwx5hy8z&dl=0",
    "MedViT_base": "https://dl.dropbox.com/scl/fi/q5c0u515dd4oc8j55bhi9/MedViT_base.pth?rlkey=5duw3uomnsyjr80wykvedjhas&st=incconx4&dl=0",
    "MedViT_large": "https://dl.dropbox.com/scl/fi/owujijpsl6vwd481hiydd/MedViT_large.pth?rlkey=cx9lqb4a1288nv4xlmux13zoe&st=kcehwbrb&dl=0"
}

def download_checkpoint(url, path):
    print(f"Downloading checkpoint from {url}...")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
    print(f"Checkpoint downloaded and saved to {path}")


In [ ]:
# Define the non-MNIST training routine
def specificity_per_class(conf_matrix):
    """Calculates specificity for each class."""
    specificity = []
    for i in range(len(conf_matrix)):
        tn = conf_matrix.sum() - (conf_matrix[i, :].sum() + conf_matrix[:, i].sum() - conf_matrix[i, i])
        fp = conf_matrix[:, i].sum() - conf_matrix[i, i]
        specificity.append(tn / (tn + fp))
    return specificity

def overall_accuracy(conf_matrix):
    """Calculates overall accuracy for multi-class."""
    tp_tn_sum = conf_matrix.trace()  # Sum of all diagonal elements (TP for all classes)
    total_sum = conf_matrix.sum()  # Sum of all elements in the matrix
    return tp_tn_sum / total_sum


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix, cohen_kappa_score, classification_report
from sklearn.preprocessing import label_binarize
import sys
from tqdm import tqdm
import torch.optim as optim

state={}
    
def train(epochs, net, train_loader, test_loader, optimizer, scheduler, loss_function, device):
    best_acc = 0.0
    for epoch in range(epochs):
        net.train()
        running_loss = 0.0
        train_bar = tqdm(train_loader, file=sys.stdout) 
        # Training Loop
        for step, datax in enumerate(train_bar):
            images, localizer,labels = datax
            images = images.to(device).float()  
            labels = labels.to(device).long() 
            optimizer.zero_grad()
            outputs = net(images.to(device))
            loss = loss_function(outputs, labels.to(device))
            loss.backward()
            optimizer.step()
            scheduler.step()
            running_loss += loss.item() 
            train_bar.desc = f"train epoch[{epoch + 1}/{epochs}] loss:{loss:.3f}"

        # Validation Loop
        net.eval()
        all_preds = []
        all_labels = []
        all_probs = []  # Store raw probabilities/logits for AUC
        acc = 0.0

        with torch.no_grad():
            val_bar = tqdm(test_loader, file=sys.stdout)
            for val_data in val_bar:
                val_images, val_labels = val_data
                outputs = net(val_images.to(device))  # Raw outputs (logits)
                probs = torch.softmax(outputs, dim=1)  # Convert to probabilities

                predict_y = torch.max(probs, dim=1)[1]  # Predicted class   
                # Collect predictions, labels, and probabilities
                all_preds.extend(predict_y.cpu().numpy())
                all_labels.extend(val_labels.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())   
                # Calculate accuracy
                acc += torch.eq(predict_y, val_labels.to(device)).sum().item()

        # Calculate metrics
        val_accurate = acc / len(test_loader.dataset)
        precision = precision_score(all_labels, all_preds, average='weighted')
        recall = recall_score(all_labels, all_preds, average='weighted')  # Sensitivity
        f1 = f1_score(all_labels, all_preds, average='weighted')
        rk_corr = matthews_corrcoef(all_labels, all_preds)
        qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
        # Confusion Matrix for multi-class
        conf_matrix = confusion_matrix(all_labels, all_preds)
        specificity = specificity_per_class(conf_matrix)  # List of specificities per class
        avg_specificity = sum(specificity) / len(specificity)  # Average specificity    
        # Overall Accuracy calculation
        overall_acc = overall_accuracy(conf_matrix) 
        # One-hot encode the labels for AUC calculation
        n_classes = len(conf_matrix)
        all_labels_one_hot = label_binarize(all_labels, classes=list(range(n_classes))) 
        try:
            # Compute AUC for multi-class
            auc = roc_auc_score(all_labels_one_hot, all_probs, multi_class='ovr')
        except ValueError:
            auc = float('nan')  # Handle edge case where AUC can't be computed  
        # Print metrics
        print(f'[epoch {epoch + 1}] train_loss: {running_loss / len(train_loader):.3f} '
              f'val_accuracy: {val_accurate:.4f} precision: {precision:.4f} '
              f'recall: {recall:.4f} specificity: {avg_specificity:.4f} '
              f'f1_score: {f1:.4f} auc: {auc:.4f} overall_accuracy: {overall_acc:.4f} '
              f'mcc: {rk_corr:.4f} qwk: {qwk:.4f}')

        print(f'lr: {scheduler.get_last_lr()[-1]:.8f}')
        print('\nClassification Report:')
        print(classification_report(all_labels, all_preds))
        print(f'lr: {scheduler.get_last_lr()[-1]:.8f}')

        # Save best model
        if val_accurate > best_acc:
            print('\nSaving checkpoint...')
            best_acc = val_accurate
            state = {
                'model': net.state_dict(),
                'optimizer': optimizer.state_dict(),
                'lr_scheduler': scheduler.state_dict(),
                'acc': best_acc,
                'epoch': epoch,
            }
            torch.save(state,'state.pt')
    print('Finished Training')  



In [ ]:
model_name='MedViT_small'
pretrained=True
lr = 2e-4


val_num = len(val_ds)
train_num = len(train_ds)
epochs=20
eta = epochs * train_num // batch_size


if model_name in model_classes:
    model_class = model_classes[model_name]
    net = model_class(num_classes=nb_classes).cuda()
    if pretrained:
        checkpoint_url = model_urls.get(model_name)
        if not checkpoint_url:
            raise ValueError(f"Checkpoint URL for model {model_name} not found.")
        download_checkpoint(checkpoint_url, f'./{model_name}.pth')
        checkpoint_path = f'./{model_name}.pth'
        checkpoint = torch.load(checkpoint_path)
        state_dict = net.state_dict()
        for k in ['proj_head.0.weight', 'proj_head.0.bias']:
            if k in checkpoint and checkpoint[k].shape != state_dict[k].shape:
                print(f"Removing key {k} from pretrained checkpoint")
                del checkpoint[k]
        net.load_state_dict(checkpoint, strict=False)

optimizer = optim.AdamW(net.parameters(), lr=lr, betas=[0.9, 0.999], weight_decay=0.05)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=eta, eta_min=5e-6)
#class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss()

    

In [ ]:
train(epochs, net, train_loader, val_loader, optimizer, scheduler, loss_function=criterion,device=device)

In [ ]:
def evaluate(net, data_loader, device):
    net.eval()

    all_preds = []
    all_labels = []
    all_probs = []
    acc = 0.0

    with torch.no_grad():
        eval_bar = tqdm(data_loader, file=sys.stdout)
        for data in eval_bar:
            images, labels = data
            images = images.to(device).float()
            labels = labels.to(device).long()

            outputs = net(images)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.max(probs, dim=1)[1]

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

            acc += torch.eq(preds, labels).sum().item()

    # Accuracy
    accuracy = acc / len(data_loader.dataset)

    # Metrics
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')
    mcc = matthews_corrcoef(all_labels, all_preds)
    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')

    # Confusion matrix + derived metrics
    conf_matrix = confusion_matrix(all_labels, all_preds)
    specificity = specificity_per_class(conf_matrix)
    avg_specificity = sum(specificity) / len(specificity)
    overall_acc = overall_accuracy(conf_matrix)

    # AUC (multi-class)
    n_classes = len(conf_matrix)
    all_labels_one_hot = label_binarize(all_labels, classes=list(range(n_classes)))

    try:
        auc = roc_auc_score(all_labels_one_hot, all_probs, multi_class='ovr')
    except ValueError:
        auc = float('nan')

    # Print results
    print(f'Accuracy: {accuracy:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'Specificity: {avg_specificity:.4f}')
    print(f'F1-score: {f1:.4f}')
    print(f'AUC: {auc:.4f}')
    print(f'Overall Accuracy: {overall_acc:.4f}')
    print(f'MCC: {mcc:.4f}')
    print(f'QWK: {qwk:.4f}')

    print('\nClassification Report:')
    print(classification_report(all_labels, all_preds))

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'specificity': avg_specificity,
        'f1': f1,
        'auc': auc,
        'overall_accuracy': overall_acc,
        'mcc': mcc,
        'qwk': qwk,
        'confusion_matrix': conf_matrix
    }

In [ ]:
checkpoint = torch.load("state.pt", map_location=device)

net.load_state_dict(checkpoint['model'])
net.to(device)
net.eval()
evaluate(net,test_loader,device)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
#from google.colab import files
#files.download('model.pth')
import torch
torch.save(net.state_dict(), '/content/drive/MyDrive/net_loc30.pth')